In [0]:
%run ../../commonsetup/Setup-Common

### 1: Entity registry
##### Bronze is raw, append-only capture -- one row per record per ingestion, duplicates and corrections kept as-is. No dedup/MERGE here; that's Silver's job.

In [0]:
import re
from typing import Optional
from pyspark.sql import functions as F
from pyspark.sql.streaming import StreamingQueryException

# Entities this Bronze layer knows how to ingest.
GTFS_ENTITIES = [
    "trips", "routes", "stops", "agency",
    "calendar_dates", "calendar",
    "shapes", "stop_times",
]

# Entities Gold can't be built without. A downstream-readiness concept, not an
# ingestion one -- Bronze still attempts every entity; this only gates Section 4's
# "safe to trigger Silver" signal.
CRITICAL_ENTITIES = {"trips", "stop_times", "calendar", "routes"}

# Per-mode overrides (see 01_download_gtfs_static.py for why buses needs calendar_dates).
MODE_CRITICAL_OVERRIDES = {
    "buses": CRITICAL_ENTITIES | {"calendar_dates"},
    "sydneytrains": CRITICAL_ENTITIES,
}

def critical_entities_for(mode: str) -> set:
    return MODE_CRITICAL_OVERRIDES.get(mode, CRITICAL_ENTITIES)

### 2: Discover entities present per mode's landing subfolder
##### Filenames follow `entity_YYYYMMDD.ext` under `BASE_LANDING/{mode}/`, matching against the FULL stem rather than a leading-letters prefix.
##### Fixes a real bug: a leading `^([a-zA-Z]+)` regex + `f"{entity}*.*"` glob both stop at the first non-letter, so `calendar_dates_20260920.csv` resolved as `calendar` and its glob matched `calendar_dates_...` too.

In [0]:
def discover_entity(filename: str) -> Optional[str]:
    stem = filename.rsplit(".", 1)[0]
    for entity in GTFS_ENTITIES:
        if re.match(rf"^{re.escape(entity)}_\d{{8}}$", stem):
            return entity
    return None


def discover_entities_for_mode(mode: str) -> list:
    mode_landing_dir = f"{BASE_LANDING}/{mode}"
    try:
        files = [f.name for f in dbutils.fs.ls(mode_landing_dir) if not f.isDir()]
    except Exception as e:
        print(f"[{mode}] landing subfolder not readable: {e}")
        return []
    return sorted({
        entity for fname in files
        if (entity := discover_entity(fname)) is not None
    })


# (mode, entity) pairs to ingest this run.
discovered_pairs = [
    (mode, entity)
    for mode in GTFS_MODES
    for entity in discover_entities_for_mode(mode)
]

print(f"(mode, entity) pairs ready for Bronze ingestion: {discovered_pairs}")

### 3: Append-only Auto Loader stream per (mode, entity)
##### Straight append, no dedup -- two copies of the same trip_id from two drops are two rows on purpose; corrections are new rows, not overwrites.
##### Both modes write into the same `bronze_gtfs_{entity}` table via `gtfs_mode`, but get separate Auto Loader checkpoints under CHECKPOINT_BASE (checkpoint state is per-stream, not per-table).
##### Each (mode, entity) pair runs in its own try/except so one failure doesn't block the rest -- whether that failure is safe to proceed past is decided in Section 4, via CRITICAL_ENTITIES.

In [0]:
run_results = {}  # (mode, entity) -> "success" | "failed"

for mode, entity in discovered_pairs:
    mode_landing_dir = f"{BASE_LANDING}/{mode}"
    target_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_gtfs_{entity}"
    checkpoint_dir = f"{CHECKPOINT_BASE}/{mode}/bronze_gtfs_{entity}/checkpoint"
    schema_dir = f"{CHECKPOINT_BASE}/{mode}/bronze_gtfs_{entity}/schema"

    # Exact-length date placeholder avoids the calendar/calendar_dates collision above.
    glob_filter = f"{entity}_????????.*"

    print(f"Starting Bronze stream for ({mode}, {entity}) -> {target_table} (filter: {glob_filter})")

    # Declared before try so except/finally can reference it even if the
    # chain itself throws before a query object exists.
    stream_query = None

    try:
        stream_query = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", schema_dir)
            .option("cloudFiles.schemaEvolutionMode", "rescue")
            .option("cloudFiles.rescuedDataColumn", "_rescued_data")
            .option("header", "true")
            .option("pathGlobFilter", glob_filter)
            .load(f"{mode_landing_dir}/")
            .withColumn("gtfs_mode", F.lit(mode))
            .withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_path"))
            .writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_dir)
            .option("mergeSchema", "true")
            .trigger(availableNow=True)
            .toTable(target_table)
        )

        stream_query.awaitTermination()
        print(f"Finished Bronze append for: ({mode}, {entity})")
        run_results[(mode, entity)] = "success"

    except StreamingQueryException as sqe:
        print(f"Runtime failure in ({mode}, {entity}): {sqe}")
        run_results[(mode, entity)] = "failed"

    except Exception as e:
        print(f"Initialization or file error in ({mode}, {entity}): {e}")
        run_results[(mode, entity)] = "failed"

    finally:
        # isActive can throw if the query never actually started.
        if stream_query is not None:
            try:
                if stream_query.isActive:
                    stream_query.stop()
            except Exception:
                pass



### 4: Validation + downstream readiness signal
##### Row counts per (mode, entity), plus the completeness check that decides whether it's safe to trigger Silver.

In [0]:

for (mode, entity), outcome in run_results.items():
    if outcome != "success":
        print(f"({mode}, {entity}): SKIPPED (ingestion failed this run)")
        continue
    target_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_gtfs_{entity}"
    count = spark.table(target_table).filter(F.col("gtfs_mode") == mode).count()
    print(f"{target_table} [{mode}]: {count} rows")

failed_pairs = {(mode, entity) for (mode, entity), outcome in run_results.items() if outcome != "success"}
failed_critical = {(mode, entity) for (mode, entity) in failed_pairs if entity in critical_entities_for(mode)}

In [0]:
# Persisted so orchestration can check readiness without re-deriving it -- printed output disappears once the job ends.
run_log_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_run_log"
run_log_df = spark.createDataFrame(
    [
        (mode, entity, outcome, entity in CRITICAL_ENTITIES)
        for (mode, entity), outcome in run_results.items()
    ],
    schema="gtfs_mode STRING, entity STRING, outcome STRING, is_critical BOOLEAN",
).withColumn("run_timestamp", F.current_timestamp())

run_log_df.write.format("delta").mode("append").saveAsTable(run_log_table)

if failed_critical:
    raise RuntimeError(
        f"Critical (mode, entity) pairs failed this run: {sorted(failed_critical)}. "
        f"Not safe to trigger Silver/Gold until these succeed."
    )
elif failed_pairs:
    print(
        f"Non-critical (mode, entity) pairs failed this run: {sorted(failed_pairs)}. "
        f"Proceeding -- Silver/Gold for unaffected pairs can still run."
    )
else:
    print("All (mode, entity) pairs succeeded bronze batch ingestion. Safe to trigger next layer.")